# Distance past the wall: AUC and the gap angle

When `fct_batted_balls` was built, two numbers went into the README:

- raw `hit_distance` separates home runs from other fly balls and line drives with an **AUC of 0.973**
- `distance_past_wall_ft` (distance minus the wall in the ball's direction) gets **0.997**

The interpolation needs Savant's "gap" measurements at some spray angle, but Savant doesn't
publish which one. **20°** was picked because it gave the best AUC.

This notebook rebuilds that work so you can see each step.

**Be clear about what this is NOT:** no model was trained. There's no regression, no
fitted coefficients, no `.fit()`. Each AUC here scores a single column used as-is. The only
thing "learned" from data was one number, the gap angle, and section 5 asks whether it was
chosen honestly.

Sections:
1. Load the data
2. What AUC measures (a toy example by hand)
3. Raw distance
4. Distance past the wall
5. Choosing the gap angle, and doing it with a train/test split
6. What a real supervised model would add

## 1. Load the data

Read-only connection: DuckDB allows one writer, so this won't block `dbt build`. Still close it
(last cell) before running dbt, since a read-only connection keeps the file open.

Only fly balls and line drives (ground balls and popups don't leave the park). Only parks with
measured dimensions (that drops neutral sites like Tokyo and London).

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

con = duckdb.connect("../data/statcast.duckdb", read_only=True)

df = con.sql('''
    select
        f.pitch_id, f.season, v.venue_name, f.batter_stand, f.batted_ball_type,
        f.launch_speed, f.launch_angle, f.hit_distance, f.spray_angle_deg,
        f.wall_distance_at_spray_ft, f.wall_height_at_spray_ft, f.distance_past_wall_ft,
        f.is_over_fence_hr::int as hr,
        v.distance_lf_line_ft, v.distance_lf_gap_ft, v.distance_cf_ft,
        v.distance_rf_gap_ft, v.distance_rf_line_ft
    from fct_batted_balls f
    join dim_venue_seasons v using (venue_id, season)
    where f.batted_ball_type in ('fly_ball', 'line_drive')
      and f.hit_distance is not null
      and f.wall_distance_at_spray_ft is not null
''').df()

print(f"{len(df):,} fly balls and line drives, {df.hr.sum():,} over-the-fence home runs ({df.hr.mean():.1%})")
df.head()

**❓**
1. About what share of fly balls and line drives are home runs? Keep that number in mind:
   a "model" that always says "not a home run" would be right that often. That's why accuracy
   is a bad score here and why we use AUC.
2. `hr` is `is_over_fence_hr`, not `is_home_run`. Why leave out inside-the-park home runs when the
   question is about walls?

## 2. What AUC measures

**AUC = the chance that a randomly picked home run scores higher than a randomly picked
non-home run.** (Ties count as half.)

- 1.0: every home run outranks every non-home run. The score separates them perfectly.
- 0.5: a coin flip. The score knows nothing.
- AUC only cares about **order**, not the values. You can add 1,000 to every score, or square
  it, and AUC doesn't change. It doesn't need a threshold either.

A toy example: 3 home runs and 3 outs, scored by distance.

In [ ]:
toy = pd.DataFrame({
    "hit_distance": [410, 395, 380, 400, 350, 330],
    "hr":           [  1,   1,   1,   0,   0,   0],
})

# Brute force: compare every (home run, non-home run) pair
pairs = [(h, n) for h in toy.query("hr == 1").hit_distance for n in toy.query("hr == 0").hit_distance]
wins = sum(1.0 if h > n else 0.5 if h == n else 0.0 for h, n in pairs)
print(f"{wins:g} of {len(pairs)} pairs ranked correctly -> AUC = {wins / len(pairs):.3f}")
pairs

**❓** Which pairs did distance get wrong? In a real park, what could make a 400 ft ball an out
while a 380 ft ball is a home run?

Brute force needs every pair: about 16k home runs × 180k others = 3 billion comparisons. The
same number falls out of ranks (the Mann-Whitney U statistic), which is fast. Check it matches the toy:

In [ ]:
def auc(y, score):
    '''P(random positive scores higher than random negative), ties count half.'''
    y = np.asarray(y)
    ranks = pd.Series(score).rank().to_numpy()  # ties get their average rank
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    return (ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

auc(toy.hr, toy.hit_distance)

**❓ (stretch)** Why does `ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2` count the winning pairs?
Hint: if every home run ranked below every out, their ranks would be 1, 2, ..., n_pos.

## 3. Score 1: raw distance

In [ ]:
HR_COLOR, OUT_COLOR = "#eb6834", "#2a78d6"  # categorical slots 2 and 1

def compare_hist(col, xlabel, bins, ax=None):
    '''Overlaid distributions for home runs vs everything else, each scaled to its own total.'''
    ax = ax or plt.subplots(figsize=(8, 3.5))[1]
    ax.hist(df.loc[df.hr == 0, col], bins=bins, density=True, histtype="step", lw=2,
            color=OUT_COLOR, label="not a home run")
    ax.hist(df.loc[df.hr == 1, col], bins=bins, density=True, histtype="step", lw=2,
            color=HR_COLOR, label="over-the-fence home run")
    ax.set_xlabel(xlabel)
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.legend(frameon=False, loc="upper left")
    ax.set_title(f"{xlabel}: AUC {auc(df.hr, df[col]):.4f}", loc="left")
    return ax

compare_hist("hit_distance", "hit distance (ft)", bins=np.arange(150, 500, 5));

**❓**
1. Where do the two curves overlap? That overlap is where every ranking mistake lives.
2. A 400 ft ball is a home run in some places and an out in others. Name two things that decide which.

## 4. Score 2: distance past the wall

`distance_past_wall_ft = hit_distance - wall_distance_at_spray_ft`. The wall distance comes from
the ball's spray angle and that park's measurements. dbt did this in SQL. Here it is again in
numpy, so you can change the gap angle:

```
spray angle:  -45 (LF line) ... -gap (LF gap) ... 0 (CF) ... +gap (RF gap) ... +45 (RF line)
wall:          straight lines between those five measured distances
```

In [ ]:
def wall_at(d, gap):
    '''Wall distance at each ball's spray angle, interpolating Savant's 5 points with gaps at +-gap degrees.'''
    a = d.spray_angle_deg.abs().clip(upper=45)        # balls hooking past a line use the line
    left = d.spray_angle_deg < 0
    gap_dist = np.where(left, d.distance_lf_gap_ft, d.distance_rf_gap_ft)
    line_dist = np.where(left, d.distance_lf_line_ft, d.distance_rf_line_ft)
    center_to_gap = d.distance_cf_ft + (gap_dist - d.distance_cf_ft) * a / gap
    gap_to_line = gap_dist + (line_dist - gap_dist) * (a - gap) / (45 - gap)
    return np.where(a <= gap, center_to_gap, gap_to_line)

# Data check: numpy at 20 degrees should reproduce what dbt stored
ours = pd.Series(wall_at(df, 20), index=df.index)
theirs = df.wall_distance_at_spray_ft
print("largest difference from dbt's column:", (ours - theirs).abs().max(), "ft")
# .max() silently skips NaN, so also count rows where only one side is missing
print("rows missing on one side only:", (ours.isna() != theirs.isna()).sum())

**❓** Why is it worth checking that two implementations of the same formula agree, before
trusting either one? (Analytics engineers call this reconciliation.)

**A true story:** the first version of this check printed only the `.max()` line, and it said
0.0. It passed. But the AUC sweep below came back `NaN`. The cause: 130 balls with no hit
coordinates. DuckDB's `least(NULL, 45)` returns 45 (it skips NULLs), so dbt had given those balls
the right field line's distance, while numpy correctly gave NaN. `.max()` skipped the NaNs, so the
check missed them. The model was fixed, and `tests/assert_wall_needs_spray_angle.sql` now guards it.

**❓** Find another pandas or SQL function that quietly skips missing values. When is that what
you want, and when does it hide a bug?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7))
compare_hist("hit_distance", "hit distance (ft)", bins=np.arange(150, 500, 5), ax=axes[0])
compare_hist("distance_past_wall_ft", "distance past the wall (ft)", bins=np.arange(-250, 120, 5), ax=axes[1])
axes[1].axvline(0, color="gray", lw=1, ls=":")
fig.tight_layout()

**❓**
1. The AUC rose from about 0.973 to about 0.997. Put that in pairs: out of every 1,000
   (home run, non-home run) pairs, about how many does each score get wrong?
2. Why does subtracting the wall help so much? Explain it in baseball terms, not math.
3. Look at the dotted line at 0. Some home runs are to the left of it. Look at them below.

In [ ]:
cols = ["season", "venue_name", "batter_stand", "hit_distance", "spray_angle_deg",
        "wall_distance_at_spray_ft", "wall_height_at_spray_ft", "distance_past_wall_ft"]

print("Home runs that landed SHORT of the computed wall")
df.query("hr == 1").nsmallest(10, "distance_past_wall_ft")[cols]

In [ ]:
print("Non-home runs that went furthest PAST the computed wall")
df.query("hr == 0").nlargest(10, "distance_past_wall_ft")[cols]

**❓**
1. For each table, guess a reason: tracking error, a wall that bends between the measured points,
   a tall wall (look at `wall_height_at_spray_ft`), a robbed home run, a ground-rule double?
2. Which park keeps showing up? What does that tell you about a five-point straight-line wall?
3. `distance_past_wall_ft` ignores wall height. How could you use `wall_height_at_spray_ft`?
   (Hint: the ball's height when it reaches the wall depends on `launch_angle`.)

## 5. Choosing the gap angle

Try a range of angles and score each one.

In [ ]:
angles = [10, 12.5, 15, 17.5, 20, 22.5, 25, 27.5, 30, 35]
sweep = pd.DataFrame({
    "gap_angle": angles,
    "auc": [auc(df.hr, df.hit_distance - wall_at(df, g)) for g in angles],
})

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sweep.gap_angle, sweep.auc, color=OUT_COLOR, lw=2, marker="o", ms=8)
ax.set_xlabel("gap angle (degrees from center)")
ax.set_title("AUC of distance past the wall, all seasons", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
sweep.round(5)

**❓**
1. How flat is the top of the curve? Is 20 clearly better than 17.5 or 22.5, or is that noise?
   (One way to find out: the bootstrap challenge below.)
2. Real parks are curves, not five straight lines. Why might the "best" gap angle be a little
   different from where the gaps are actually measured?

### Doing it properly: train / test

The sweep above has a flaw. **Choosing the gap angle is fitting a parameter.** It was chosen
on all three seasons and then scored on those same seasons. That's grading yourself on the
exam you studied from. It's the same problem as the leakage rule in `dim_venue_seasons`, one
level up.

The honest version: pick the angle using **2024–2025 only (train)**, then score it once on
**2026 (test)**, which the choice never saw. Splitting by time copies real use: you'd pick the
angle with past seasons and apply it to the next one.

In [ ]:
train = df[df.season < 2026]
test = df[df.season == 2026]

train_sweep = pd.Series({g: auc(train.hr, train.hit_distance - wall_at(train, g)) for g in angles})
best = train_sweep.idxmax()

print(f"best angle on train (2024-25): {best} (AUC {train_sweep[best]:.5f})")
print()
print("test (2026) AUC for a few angles:")
for g in sorted({best, 20, 22.5, 30}):
    print(f"  {g:>5}: {auc(test.hr, test.hit_distance - wall_at(test, g)):.5f}")
print(f"  raw distance: {auc(test.hr, test.hit_distance):.5f}")

**❓**
1. Did the training seasons pick the same angle as the full-data sweep?
2. Did the test AUC hold up, or drop? What would a big drop from train to test have meant?
3. Here we tuned **one** number on 180k rows, so overfitting risk is tiny. Why does that risk
   grow as you tune more parameters, or as your data shrinks (say, one hitter's batted balls in
   one park)? This is the core reason the HR/PA model will be Bayesian.
4. Should the test season ever be used to *change* the angle? What happens to it as a test if you do?

### Challenge (optional): is 20 really better than 22.5?

Bootstrap: resample the rows with replacement many times, recompute both AUCs on each sample,
and look at the spread of the difference. If the middle 95% of differences includes 0, you
can't tell them apart.

In [ ]:
# rng = np.random.default_rng(0)
# diffs = []
# for _ in range(200):
#     s = df.sample(len(df), replace=True, random_state=rng.integers(1e9))
#     diffs.append(auc(s.hr, s.hit_distance - wall_at(s, 20)) - auc(s.hr, s.hit_distance - wall_at(s, 22.5)))
# np.percentile(diffs, [2.5, 50, 97.5])

## 6. What a supervised model adds

Everything above ranks balls by **one column we built by hand**. A supervised model *learns* how
to combine many columns from labeled examples:

- **Logistic regression:** P(HR) = sigmoid(b0 + b1·launch_speed + b2·launch_angle + ...). It
  learns one weight per input.
- **Gradient boosting:** many small decision trees. It can learn shapes like "home runs need
  25–35° launch angle *and* 100+ mph", which a straight-line model can't.

That's the Level 1 park-fit model, the next notebook. It will use a time-based split like
section 5, and be scored with AUC plus **calibration** (when it says 30%, do 30% of those balls
leave the park?) and log loss.

(`notebooks/04_regression_xHR.ipynb` is an early sketch of this. It imports scikit-learn, which
isn't installed in the venv yet.)

**❓ Before the next notebook:**
1. List the inputs you'd give a batted-ball HR model. For each, is it known *at contact*?
2. Should the park-fit model use `hit_distance` as an input? It already "knows" the air, altitude
   and weather of the park the ball was hit in. What goes wrong when you replay a ball hit at
   Coors Field in all 30 parks, if the input is the distance it flew at Coors?
   (Compare: `launch_speed`, `launch_angle`, `spray_angle_deg`.)
3. Distance past the wall already gets 0.997. What is left for a model to learn? Where would you
   look? (Hint: section 4's tables.)

In [ ]:
con.close()  # release the file before running dbt